# Daily H=22, notebook 2/3: busca Optuna do TSMixerX

Este notebook executa a busca na primeira janela de oito anos de treino e dois
de validação. O preset final usa 40 trials, até 400 passos, pruning e early
stopping.

**Tempo estimado no Colab com GPU:** 35–55 minutos. O teto operacional é 60
minutos, deixando 10 minutos para exportar o banco, empacotar e baixar os
resultados antes do limite de 70 minutos.

In [ ]:
# Configuração compartilhada: repositório + armazenamento persistente
from pathlib import Path
import os, shutil, subprocess, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('(Not on Colab / Drive mount skipped):', exc)

REPO_URL = 'https://github.com/hugogobato/Mestrado_Anna_Julia.git'
REPO = Path('/content/Mestrado_Anna_Julia')
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

EXP = REPO / 'experiments/daily_h22'
PERSIST = Path('/content/drive/MyDrive/Mestrado_Anna_Julia/daily_h22')
DATA_DIR = PERSIST / 'data'
RESULTS_DIR = PERSIST / 'results'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
os.environ['DAILY_H22_DATA_DIR'] = str(DATA_DIR)
os.environ['DAILY_H22_RESULTS_DIR'] = str(RESULTS_DIR)
os.environ['MPLCONFIGDIR'] = '/content/matplotlib_cache'
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(EXP / 'requirements-colab.txt')], check=True)

def run_script(name, *args):
    command = [sys.executable, str(EXP / 'src' / name), *map(str, args)]
    print('RUN:', ' '.join(command))
    subprocess.run(command, check=True, env=os.environ.copy())

print('Experiment:', EXP)
print('Persistent data:', DATA_DIR)
print('Persistent results:', RESULTS_DIR)
try:
    import torch
    print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
          '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
except Exception as exc:
    print('Torch check failed:', exc)


## Configuração

O modo final foi dimensionado para concluir em uma sessão de até 70 minutos.
Ele continua sendo uma busca de hiperparâmetros com seleção conjunta do número
de features, mas usa um orçamento computacional compatível com o Colab.

In [ ]:
RUN_MODE = 'final'  # 'smoke', 'validation' ou 'final'
SESSION_BUDGET_MINUTES = 60
TARGET_TRIALS = {'smoke': 3, 'validation': 15, 'final': 40}[RUN_MODE]
MAX_STEPS = {'smoke': 30, 'validation': 200, 'final': 400}[RUN_MODE]
LOCAL_DB = Path('/content/tsmixerx_daily.db')
BACKUP_DIR = RESULTS_DIR / 'hp_search/sqlite_backup'
print(RUN_MODE, TARGET_TRIALS, MAX_STEPS, BACKUP_DIR)


## Executar ou retomar a busca

O objetivo é o QLIKE médio dos 22 horizontes na validação. O pruning chama
`trial.report` e `trial.should_prune` em cada checagem. O target é treinado em
log-variância para impedir previsões negativas ou praticamente nulas.

In [ ]:
args = [
    '--target-trials', TARGET_TRIALS,
    '--max-steps', MAX_STEPS,
    '--timeout-minutes', SESSION_BUDGET_MINUTES,
    '--storage', LOCAL_DB,
    '--backup-dir', BACKUP_DIR,
]
if RUN_MODE == 'smoke':
    args += ['--smoke']
run_script('12_tsmixerx_optuna.py', *args)


## Estado da busca e melhor configuração

In [ ]:
import json, pandas as pd
best_path = RESULTS_DIR / 'hp_search/best_config.json'
if best_path.exists():
    best = json.loads(best_path.read_text())
    print(json.dumps(best, indent=2))
trial_files = sorted((RESULTS_DIR / 'hp_search').glob('trials_*.csv'))
if trial_files:
    trials = pd.read_csv(trial_files[-1])
    display(trials.tail(20))
    print(trials['state'].value_counts(dropna=False))
    if RUN_MODE == 'final' and len(trials) < TARGET_TRIALS:
        raise RuntimeError('Busca incompleta: o teto de 60 minutos foi atingido inesperadamente.')


In [ ]:
ARCHIVE_NAME = 'daily_h22_notebook2_hp_state'
# Empacotamento e download seguro do estado atual
import shutil
from pathlib import Path

archive_base = Path('/content') / ARCHIVE_NAME
output_file = shutil.make_archive(str(archive_base), 'zip', root_dir=PERSIST)
print('Archive created:', output_file)
try:
    from google.colab import files
    files.download(output_file)
    print("Downloaded:", output_file)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
